# CATS Dataset Audit

## Controlled Anomalies Time Series (CATS)

This notebook performs the initial audit of the publicly available CATS (Controlled Anomalies Time Series) Dataset.

The purpose of this phase is to understand and verify the structure, temporal properties, signals, labels, and anomaly metadata of the dataset before any feature engineering or anomaly-detection modelling begins.

### Audit Questions

1. What are the exact columns?
2. How many rows are there?
3. What is the timestamp range?
4. Is the timestamp actually 1-second apart?
5. Are there duplicate timestamps?
6. Are there missing timestamps?
7. Which columns are the 17 signals?
8. What are the data types?
9. What is the distribution of `y`?
10. What are the actual `category` values?
11. What does `metadata.csv` contain?
12. How many anomaly segments are there?
13. What are the actual anomaly categories?
14. Which channels are identified as root causes?
15. Which channels are affected?

### Important

This notebook is an **audit only**.

At this stage, we will not:
- train an anomaly-detection model;
- perform feature engineering;
- randomly split the time series;
- tune model parameters;
- modify the raw dataset;
- use ground-truth labels as model features.

The raw CATS data will be treated as read-only.

In [1]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw CATS data
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"
METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "metadata.csv"

print("Project root:", PROJECT_ROOT)
print("Data file:", DATA_PATH)
print("Metadata file:", METADATA_PATH)

Project root: c:\Users\ADEGOKE\Desktop\CATS
Data file: c:\Users\ADEGOKE\Desktop\CATS\data\raw\cats\data.parquet
Metadata file: c:\Users\ADEGOKE\Desktop\CATS\data\raw\metadata.csv


## 1 Inspect the Parquet schema without loading the full dataset


In [8]:
import pyarrow.parquet as pq

# Read only the parquet schema
parquet_file = pq.ParquetFile(DATA_PATH)

print("Number of columns:", len(parquet_file.schema.names))
print("\nExact columns:")

for i, column in enumerate(parquet_file.schema.names, start=1):
    print(f"{i}.{column}")

Number of columns: 20

Exact columns:
1.aimp
2.amud
3.arnd
4.asin1
5.asin2
6.adbr
7.adfl
8.bed1
9.bed2
10.bfo1
11.bfo2
12.bso1
13.bso2
14.bso3
15.ced1
16.cfo1
17.cso1
18.y
19.category
20.__index_level_0__


### Question 1 — Conclusion

The CATS Parquet file contains 20 columns.

- 17 columns are system signals.
- `y` is the anomaly label.
- `category` identifies the anomaly category.
- `__index_level_0__` is an index column and is not a modelling feature.

The exact role and meaning of the 17 signal columns will be verified from the CATS documentation and metadata rather than inferred from their abbreviated names.

## 2. Number of Rows

In [9]:
# Get the exact number of rows from the Parquet metadata
num_rows = parquet_file.metadata.num_rows
print("Number of rows:", num_rows)

Number of rows: 5000000


### Question 2 — Conclusion

The CATS Parquet file contains exactly 5,000,000 rows.

This matches the official CATS Version 2 documentation, which states that the dataset contains 5 million timestamps.

## 3. Timestamp Range

In [12]:
# Inspect the Parquet metadata to see how the index is stored

metadata = parquet_file.metadata.metadata

print("Parquet metadata keys:")
for key in metadata:
    print(key)

Parquet metadata keys:
b'pandas'
b'ARROW:schema'


In [13]:
import json

pandas_metadata = metadata[b"pandas"]
pandas_metadata = json.loads(pandas_metadata.decode("utf-8"))

print(json.dumps(pandas_metadata, indent=2))

{
  "index_columns": [
    "__index_level_0__"
  ],
  "column_indexes": [
    {
      "name": null,
      "field_name": null,
      "pandas_type": "unicode",
      "numpy_type": "object",
      "metadata": {
        "encoding": "UTF-8"
      }
    }
  ],
  "columns": [
    {
      "name": "aimp",
      "field_name": "aimp",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "amud",
      "field_name": "amud",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "arnd",
      "field_name": "arnd",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "asin1",
      "field_name": "asin1",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "metadata": null
    },
    {
      "name": "asin2",
      "field_name": "asin2",
      "pandas_type": "float64",
      "numpy_type": "float64",
      "meta

In [14]:
# Inspect the timestamp index

df_index = pd.read_parquet(DATA_PATH).index

print("First timestamp:", df_index[0])
print("Last timestamp:", df_index[-1])
print("Data type:", df_index.dtype)

First timestamp: 2023-01-01 00:00:00
Last timestamp: 2023-02-27 20:53:19
Data type: datetime64[us]


### Question 3 — Conclusion

The CATS dataset covers the period from **2023-01-01 00:00:00** to **2023-02-27 20:53:19**.

The dataset contains 5,000,000 observations sampled at 1 Hz (one observation per second), which corresponds to approximately 57.87 days of continuous data.

The timestamp is stored as the DataFrame index with datetime type `datetime64[us]`.

## Question 4 — Is the timestamp actually 1 second apart?

We will check the differences between consecutive timestamps to verify the sampling interval.